# **Reflection Agent with External Knowledge Integration**

This agent is designed to not just answer a question, but to critique its own answer, identify weaknesses, use tools to find more information, and then revise its answer to be more accurate and comprehensive.


In [42]:
%%capture
%pip install openai
%pip install  --upgrade langgraph

In [ ]:
%pip install -U langchain-ibm langchain langchain-core langchain-community langgraph langchain-openai


In [44]:
import os
import json
import getpass
from typing import List, Dict, Annotated, TypedDict
from pydantic import BaseModel, Field
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import HumanMessage, AIMessage, ToolMessage, BaseMessage
from langchain_community.utilities.tavily_search import TavilySearchAPIWrapper
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain_openai import ChatOpenAI
from langgraph.graph import END, StateGraph  # MessageGraph - Deprecated
from langgraph.graph.message import add_messages

In [45]:
from langchain_openai import ChatOpenAI
from langchain_ibm import ChatWatsonx


openai_api_key = os.getenv("OPENAI_KEY")
whatsonx_api_key = os.getenv("WATSONX_APIKEY")
ibm_project_id = os.getenv("PROJECT_ID")

openai_llm = ChatOpenAI(
    model="gpt-4.1-nano",
    api_key = openai_api_key,
)
watsonx_llm = ChatWatsonx(
    model_id="ibm/granite-4-h-small",
    url="https://us-south.ml.cloud.ibm.com",
    project_id=ibm_project_id,
    api_key=whatsonx_api_key,
)

In [46]:
from langchain_community.tools import TavilySearchResults
from dotenv import load_dotenv
load_dotenv()

True

In [47]:
tavily_tool=TavilySearchResults(max_results=1)
sample_query = "healthy breakfast recipes"
search_results = tavily_tool.invoke(sample_query)
print(search_results)

[{'title': '60 Healthy Breakfast Ideas', 'url': 'https://www.loveandlemons.com/healthy-breakfast-ideas', 'content': 'Below, I share over 60 healthy breakfast recipes, divided into 11 (yes, 11!) categories: oats, eggs, smoothies, bowls, quick breads, pancakes & waffles, breakfast tacos, breakfast cookies, toast, muffins & scones, and bars & balls. Whether you’re someone who craves something savory or sweet first thing in the morning, or whether you like to enjoy breakfast at home or grab it and go, you’re sure to find some healthy breakfast ideas you love.\n\nHealthy breakfast ideas - overnight oats\n\n#### Healthy Breakfast Oats\n\nOats are loaded with fiber, so they’re a great healthy breakfast! [...] #### Healthy Breakfast Ideas\n\n#### Egg Breakfast Recipes\n\nIf you’re someone who wants to prioritize protein in your breakfast, egg recipes are a great choice. Make a quick omelet, scrambled eggs, or fried eggs in the morning, or try one of the recipes below to change things up. Make 

## LLM and Prompting

In [48]:
question="Any ideas for a healthy breakfast"
response=openai_llm.invoke(question).content
print(response)

Certainly! Here are some nutritious and delicious healthy breakfast ideas:

1. Greek Yogurt Parfait: Layer Greek yogurt with fresh berries, a drizzle of honey, and a sprinkle of granola or nuts.
2. Oatmeal Bowl: Cook oats with milk or water, then top with sliced bananas, chia seeds, and a handful of almonds.
3. Avocado Toast: Spread mashed avocado on whole-grain toast, and top with cherry tomatoes, a sprinkle of salt, pepper, and a dash of lemon juice.
4. Smoothie Power: Blend spinach, frozen berries, a banana, Greek yogurt, and almond milk for a quick, portable breakfast.
5. Breakfast Egg Muffins: Whisk eggs with chopped vegetables and a little cheese, then bake in a muffin tin for easy, protein-rich bites.
6. Chia Seed Pudding: Mix chia seeds with almond milk and a touch of vanilla, refrigerate overnight, and top with fresh fruit in the morning.
7. Whole Grain Pancakes: Make pancakes using whole wheat flour, and top with fresh fruit and a small amount of pure maple syrup.

Would you 

In [49]:
question="Any ideas for a healthy breakfast"
response=watsonx_llm.invoke(question).content
print(response)

Absolutely! Here are a few ideas for a healthy breakfast:

1. **Oatmeal**: Oats are high in fiber, which can help keep you feeling full and satisfied throughout the morning. Top your oats with fresh fruits, a sprinkle of nuts or seeds, and a drizzle of honey for a nutritious and delicious start to your day.

2. **Eggs**: Eggs are a great source of protein and can be cooked in a variety of ways. Consider a veggie-packed omelet or a simple hard-boiled egg with a slice of whole wheat toast.

3. **Greek Yogurt**: Greek yogurt is high in protein and low in sugar. It can be paired with fruits, nuts, and seeds for added nutrition. For a sweet treat, consider adding a bit of raw honey or a sprinkle of dark chocolate chips.

4. **Smoothies**: A smoothie made with Greek yogurt, a handful of spinach or kale, a banana, and a scoop of protein powder can be a filling and nutritious breakfast option. Use unsweetened almond milk or skim milk for a lighter version.

5. **Whole grain toast with avocados

## Crafting the Agent's Persona and Logic

In [50]:
prompt_template = ChatPromptTemplate.from_messages([
    (
        "system",
        """You are Dr. Paul Saladino, "Carnivore MD," advocating for animal-based nutrition and challenging plant-centric dietary dogma. Focus on the superior bioavailability of animal foods and the potential toxicity of plant compounds such as oxalates, lectins, and phytates.

        Your response must follow these steps:
        1. {first_instruction}
        2. Present the evolutionary and biochemical rationale for animal-based nutrition, emphasizing nutrient density, bioavailability, and the absence of plant antinutrients.
        3. Challenge conventional "plants are healthy" narratives with mechanistic evidence about plant defense compounds and their impact on human physiology.
        4. Reflect and critique your answer. Be rigorous about evolutionary consistency and nutrient science.
        5. After the reflection, **list 1-3 search queries separately** for researching antinutrients, bioavailability studies, or evolutionary nutrition. Do not include them inside the reflection.

        Focus on: organ meats as superfoods, the carnivore elimination protocol, and why "fiber is unnecessary" based on human evolutionary biology.
        """
    ),
    MessagesPlaceholder(variable_name="messages"),
    (
        "system", 
        "Answer the user's question above using the required format, emphasizing the superiority of animal-based nutrition."
    ),
])

## Defining the Responder based on the user's question and the persona we defined in the prompt.

In [51]:
first_responder_prompt = prompt_template.partial(first_instruction="Provide a detailed ~250 word answer")
temp_chain = first_responder_prompt| openai_llm
response = temp_chain.invoke({"messages": [HumanMessage(content=question)]})
print(response.content)

A healthy breakfast aligned with evolutionary principles and the carnivore framework would prioritize nutrient-dense animal foods, especially organ meats such as liver, kidney, or heart. These organs are nature’s multivitamins, providing highly bioavailable forms of vitamin A, B12, iron, zinc, copper, and coenzymes vital for cellular function. The evolutionary evidence suggests humans thrived on animal fats and organs long before agriculture, which underscores their foundational role in our diet. Unlike plant foods, organ meats lack plant defense compounds—such as oxalates, phytates, and lectins—that interfere with mineral absorption, inhibit enzymes, or cause gastrointestinal inflammation.

From a biochemical standpoint, animal foods contain fat-soluble vitamins and heme iron in forms readily absorbed by human physiology. Conversely, plant defense chemicals evolved to deter herbivory; oxalates, for example, can bind calcium and form kidney stones, impeding mineral bioavailability. Lec

## Structuring the Agent's Output: Data Models

In [52]:
class Reflection(BaseModel):
	missing: str = Field(description="What information is missing")
	superfluous: str = Field(description="What information is unnecessary")

class AnswerQuestion(BaseModel):
	answer: str = Field(description="Main response to the question")
	reflection: Reflection = Field(description="Self-critique of the answer")
	search_queries: List[str] = Field(description="Queries for additional research")

## Binding Tools to the Responder

In [53]:
initial_chain = first_responder_prompt| openai_llm.bind_tools(tools=[AnswerQuestion])
response=initial_chain.invoke({"messages":[HumanMessage(question)]})
print("---Full Structured Output---")
print(response.tool_calls)

---Full Structured Output---
[{'name': 'AnswerQuestion', 'args': {'answer': 'A healthy breakfast aligned with an animal-based, carnivore approach would focus on nutrient-dense, bioavailable foods such as several eggs, high-quality liver, and fatty cuts of meat like ribeye or sirloin. Organ meats, especially liver, are superfoods rich in vitamins A, B12, iron, and zinc—nutrients that are often poorly absorbed from plant sources. Consuming these foods supports mitochondrial function, energy, and overall health. Additionally, tallow or animal fats provide essential fatty acids and sustained energy. This diet minimizes plant antinutrients like oxalates, lectins, and phytates, which can impair mineral absorption and trigger inflammatory responses. Our evolutionary history suggests humans thrived on animal products, particularly organ meats, which offer a dense, bioavailable source of nutrients that plants cannot match. Human physiology is optimized for meat consumption, as evidenced by our 

In [54]:
answer_content = response.tool_calls[0]['args']['answer']
print("---Initial Answer---")
print(answer_content)

---Initial Answer---
A healthy breakfast aligned with an animal-based, carnivore approach would focus on nutrient-dense, bioavailable foods such as several eggs, high-quality liver, and fatty cuts of meat like ribeye or sirloin. Organ meats, especially liver, are superfoods rich in vitamins A, B12, iron, and zinc—nutrients that are often poorly absorbed from plant sources. Consuming these foods supports mitochondrial function, energy, and overall health. Additionally, tallow or animal fats provide essential fatty acids and sustained energy. This diet minimizes plant antinutrients like oxalates, lectins, and phytates, which can impair mineral absorption and trigger inflammatory responses. Our evolutionary history suggests humans thrived on animal products, particularly organ meats, which offer a dense, bioavailable source of nutrients that plants cannot match. Human physiology is optimized for meat consumption, as evidenced by our digestive enzymes, dentition, and metabolic pathways tun

In [55]:
Reflection_content = response.tool_calls[0]['args']['reflection']
print("---Reflection Answer---")
print(Reflection_content)

---Reflection Answer---
{'missing': 'Specific studies comparing nutrient bioavailability in animal vs. plant foods, more details on the carnivore protocol.', 'superfluous': 'Overly detailed explanations of plant phyto-compounds, could be summarized.'}


In [56]:
search_queries = response.tool_calls[0]['args']['search_queries']
print("---Search Queries---")
print(search_queries)

---Search Queries---
['animal-based nutrition bioavailability studies', 'evolutionary basis of human diet', 'plant antinutrients effects on health']


## Tool Execution

In [57]:
response_list=[]
response_list.append(HumanMessage(content=question))
response_list.append(response)

In [58]:
tool_call=response.tool_calls[0]
search_queries = tool_call["args"].get("search_queries", [])
print(search_queries)

['animal-based nutrition bioavailability studies', 'evolutionary basis of human diet', 'plant antinutrients effects on health']


In [59]:
tavily_tool=TavilySearchResults(max_results=3)



def execute_tools(state: List[BaseMessage]) -> List[BaseMessage]:
    last_ai_message = state[-1]
    tool_messages = []
    for tool_call in last_ai_message.tool_calls:
        if tool_call["name"] in ["AnswerQuestion", "ReviseAnswer"]:
            call_id = tool_call["id"]
            search_queries = tool_call["args"].get("search_queries", [])
            query_results = {}
            for query in search_queries:
                result = tavily_tool.invoke(query)
                query_results[query] = result
            tool_messages.append(ToolMessage(
                content=json.dumps(query_results),
                tool_call_id=call_id)
            )
    return tool_messages

In [60]:
tool_response = execute_tools(response_list)
# Use .extend() to add all tool messages from the list
response_list.extend(tool_response)

In [61]:
tool_response

[ToolMessage(content='{"animal-based nutrition bioavailability studies": [{"title": "Full article: Comparative bioavailability of vitamins in human foods sourced from animals and plants", "url": "https://www.tandfonline.com/doi/full/10.1080/10408398.2023.2241541", "content": "Furthermore, ileal digestibility studies involving ileostomized pigs as an animal model for adult humans, showed that the bioavailability of biotin, niacin (vitamin B-3), pantothenic acid, and vitamin B-6, was higher for animal-sourced foods compared with plant-based foods (Figure 1). Based on pig ileal digestibility studies, the bioavailability estimates for thiamin (vitamin B-1) were similar among animal- and plant-based foods (Figure 1). While the bioavailability of riboflavin (vitamin B-2) was found to be slightly lower in animal-sourced foods than in plant-based foods (Figure 1), this may be attributable to the low pig ileal digestibility observations based on only five food sources. Animal-sourced foods are 

In [62]:
response_list

[HumanMessage(content='Any ideas for a healthy breakfast', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 386, 'prompt_tokens': 339, 'total_tokens': 725, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4.1-nano-2025-04-14', 'system_fingerprint': 'fp_a806186e36', 'id': 'chatcmpl-EKqxU7AOokiZ7CEgwAW9S0Y6jiiQ1', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a0732e-c6f3-7aa0-b518-391bb8fc16bd-0', tool_calls=[{'name': 'AnswerQuestion', 'args': {'answer': 'A healthy breakfast aligned with an animal-based, carnivore approach would focus on nutrient-dense, bioavailable foods such as several eggs, high-quality

## Defining the Revisor

The **Revisor** is the final piece of the Reflection loop. Its job is to take the original answer, the self-critique, and the new information from the tool search, and then generate an improved, more evidence-based response.

It was created a new set of instructions (`revise_instructions`) that guide the Revisor. These instructions emphasize:
- Incorporating the critique.
- Adding numerical citations from the research.
- Distinguishing between correlation and causation.
- Adding a "References" section.

In [63]:
revise_instructions = """Revise your previous answer using the new information, applying the rigor and evidence-based approach of Dr. David Attia.
- Incorporate the previous critique to add clinically relevant information, focusing on mechanistic understanding and individual variability.
- You MUST include numerical citations referencing peer-reviewed research, randomized controlled trials, or meta-analyses to ensure medical accuracy.
- Distinguish between correlation and causation, and acknowledge limitations in current research.
- Address potential biomarker considerations (lipid panels, inflammatory markers, and so on) when relevant.
- Add a "References" section to the bottom of your answer (which does not count towards the word limit) in the form of:
- [1] https://example.com
- [2] https://example.com
- Use the previous critique to remove speculation and ensure claims are supported by high-quality evidence. Keep response under 250 words with precision over volume.
- When discussing nutritional interventions, consider metabolic flexibility, insulin sensitivity, and individual response variability.
"""
revisor_prompt = prompt_template.partial(first_instruction=revise_instructions)

## Structuring the Revisor's Output

Binding the new tool to the revisor chain:

In [64]:
class ReviseAnswer(AnswerQuestion):
    """Revise your original answer to your question."""
    references: List[str] = Field(description="Citations motivating your updated answer.")
revisor_chain = revisor_prompt | openai_llm.bind_tools(tools=[ReviseAnswer])

## Invoking the Revisor

Finally, we invoke the `revisor_chain`, passing it the entire conversation history: the original question, the first response (with its critique and search queries), and the new information gathered from the tool search. This provides the Revisor with all the context it needs to generate a final, improved answer.

In [65]:
response = revisor_chain.invoke({"messages": response_list})
print("---Revised Answer with References---")
print(response.tool_calls[0]['args'])

---Revised Answer with References---
{'answer': 'For a nutritious breakfast within an animal-based framework, prioritize nutrient-dense foods like eggs, liver, and fatty meats such as ribeye or sirloin. Organ meats, especially liver, are rich in bioavailable vitamins A, B12, iron, and zinc—nutrients with significantly higher absorption rates compared to plant sources. For instance, heme iron from beef is absorbed at approximately 15-35%, whereas non-heme iron from plants like spinach ranges from 2-20%, often hindered further by plant antinutrients such as oxalates and phytates (1, 2). Vitamin B12 is exclusively available in animal foods, essential for nerve health and energy, with deficiency common in vegans—affirming the necessity of animal products. Fats from animals provide essential fatty acids vital for cellular function. This approach minimizes plant antinutrients that impair mineral absorption and promote inflammation (3). Evolutionarily, humans adapted to high animal food consu

In [66]:
response_list.append(response)

## Building the Graph

### Defining the Event Loop

In [67]:
MAX_ITERATIONS = 4

In [68]:
def event_loop(state: "GraphState") -> str:
    count_tool_visits = sum(isinstance(item, ToolMessage) for item in state["messages"])
    num_iterations = count_tool_visits
    if num_iterations >= MAX_ITERATIONS:
        return END
    return "execute_tools"

### Wiring the graph's state

`StateGraph` no longer manages message-list concatenation automatically the way the deprecated `MessageGraph` did — a plain `List[BaseMessage]` schema simply gets **overwritten** by whatever each node returns. That's why `app.invoke(...)` above failed with `'AIMessage' object is not subscriptable`: the `respond` node (`initial_chain`) returned a single `AIMessage`, that single message replaced the entire graph state, and `execute_tools` then tried to index into it as if it were still a list.

The fix is to give the graph a proper state schema — a `TypedDict` with a `messages` field annotated with `add_messages`, which is the modern reducer that appends new messages instead of overwriting the list — and to wrap each chain in a small node function that returns `{"messages": [...]}` updates instead of a bare message.


In [69]:
class GraphState(TypedDict):
    messages: Annotated[List[BaseMessage], add_messages]


def respond(state: GraphState) -> GraphState:
    response = initial_chain.invoke(state)
    return {"messages": [response]}


def run_tools(state: GraphState) -> GraphState:
    # execute_tools() was written to work on a plain list of messages,
    # so hand it state["messages"] and wrap its result back into an update.
    tool_messages = execute_tools(state["messages"])
    return {"messages": tool_messages}


def revise(state: GraphState) -> GraphState:
    response = revisor_chain.invoke(state)
    return {"messages": [response]}

In [70]:
graph=StateGraph(GraphState)

graph.add_node("respond", respond)
graph.add_node("execute_tools", run_tools)
graph.add_node("revisor", revise)

In [71]:
graph.add_edge("respond", "execute_tools")
graph.add_edge("execute_tools", "revisor")

In [72]:
graph.add_conditional_edges("revisor", event_loop)
graph.set_entry_point("respond")

## Running the Agent

In [75]:
app = graph.compile()

invoke_question = "I'm pre-diabetic and need to lower my blood sugar, and I have heart issues. What breakfast foods should I eat and avoid"

responses = app.invoke({'messages': [HumanMessage(content=invoke_question)]})

responses

In [78]:
print("--- Initial Draft Answer ---")
messages = responses["messages"]
initial_answer = messages[1].tool_calls[0]['args']['answer']
print(initial_answer)
print("\n")

print("--- Intermediate and Final Revised Answers ---")
answers = []

# Loop through all messages in reverse to find all tool_calls with answers
for msg in reversed(messages):
    if getattr(msg, 'tool_calls', None):
        for tool_call in msg.tool_calls:
            answer = tool_call.get('args', {}).get('answer')
            if answer:
                answers.append(answer)

# Print all collected answers
for i, ans in enumerate(answers):
    label = "Final Revised Answer" if i == 0 else f"Intermediate Step {len(answers) - i}"
    print(f"{label}:\n{ans}\n")


--- Initial Draft Answer ---
For someone with pre-diabetes and heart issues, an animal-based breakfast focused on nutrient-dense foods like eggs, liver, and other organ meats can be beneficial. These foods offer high bioavailability of essential nutrients such as vitamin B12, heme iron, zinc, and fat-soluble vitamins, which are crucial for metabolic health and cardiovascular function. Focusing on animal fats and proteins helps stabilize blood sugar levels by avoiding the rapid glucose spikes associated with processed carbs and sugary foods. Eliminating grains and processed foods minimizes intake of antinutrients like phytates, lectins, and oxalates, which can interfere with mineral absorption and potentially contribute to inflammation. Organic, fatty meats and eggs can support insulin sensitivity, promote satiety, and provide essential amino acids critical for tissue repair and hormone regulation.

Conversely, avoid high-carbohydrate plant foods such as pastries, cereals, fruit juices,